# 09 – Model: Competition Prediction

### Purpose
Prediction of the number of offers using ML models.

### Steps
- Feature-Auswahl
- Modelltraining
- Evaluation
- Feature Importance


--------------------
#### Imports & Setup
-------------------

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import pickle
import json

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

# importing modules for data preparation
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# importing modules for training
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score, confusion_matrix

# importing modules for pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# importing modules for hyperparameter optimization and comparison of models
from sklearn.model_selection import validation_curve, learning_curve


In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.visualization import (plot_roc_curve, plot_pr_curve, plot_confusion_matrix,
                            plot_feature_importance, visual_eu, visual_de)

from my_scripts.eda import (overview, filter_germany)

In [5]:
# ---------------------------------------------------------
# Load data and artifacts
# ---------------------------------------------------------

BASE_DIR = Path().resolve().parent
DATA_PATH = BASE_DIR / "data" / "dataset_nlp.pkl"
MODELS_DIR = BASE_DIR / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "risk_model.pkl"
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.pkl"
FEATURE_LIST_PATH = MODELS_DIR / "feature_list.json"
METRICS_PATH = MODELS_DIR / "metrics.json"

In [6]:
# Load dataset
with open(DATA_PATH, "rb") as f:
    df = pickle.load(f)

------------------------
# Modelling

-----------------

In [ ]:
# ---------------------------------------------------------
# Create feature and test
# ---------------------------------------------------------

# Load feature info
TARGET_COL = "IS_FAILED_TENDER"

features = df.drop(columns=[TARGET_COL])
target = df[TARGET_COL]

In [24]:
# Train/test split

features_train, features_test, target_train, target_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target
)

In [ ]:
# save features_test as 'features_test.csv'
features_test.to_csv(BASE_DIR / "data" / "features_test.csv", index=False)

--------------------
## Base Model

--------------------

In [ ]:
# Identify feature types

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['YEAR', 'ID_TYPE', 'CANCELLED', 'CORRECTIONS', 'B_DYN_PURCH_SYST', 'B_ACCELERATED', 'OUT_OF_DIRECTIVES', 'CRIT_PRICE_WEIGHT', 'B_ELECTRONIC_AUCTION', 'NUMBER_AWARDS', 'B_AWARDED_TO_A_GROUP', 'B_CONTRACTOR_SME', 'AWARD_QUARTER', 'DAYS_TO_AWARD', 'IS_LOW_COMPETITION', 'HAS_MULTIPLE_LOTS', 'VALUE_EURO_MISSING', 'AWARD_VALUE_EURO_MISSING', 'NUMBER_OFFERS_MISSING', 'NLP_SVD_0', 'NLP_SVD_1', 'NLP_SVD_2', 'NLP_SVD_3', 'NLP_SVD_4', 'NLP_SVD_5', 'NLP_SVD_6', 'NLP_SVD_7', 'NLP_SVD_8', 'NLP_SVD_9', 'NLP_SVD_10', 'NLP_SVD_11', 'NLP_SVD_12', 'NLP_SVD_13', 'NLP_SVD_14', 'NLP_SVD_15', 'NLP_SVD_16', 'NLP_SVD_17', 'NLP_SVD_18', 'NLP_SVD_19', 'NLP_SVD_20', 'NLP_SVD_21', 'NLP_SVD_22', 'NLP_SVD_23', 'NLP_SVD_24', 'NLP_SVD_25', 'NLP_SVD_26', 'NLP_SVD_27', 'NLP_SVD_28', 'NLP_SVD_29', 'NLP_SVD_30', 'NLP_SVD_31', 'NLP_SVD_32', 'NLP_SVD_33', 'NLP_SVD_34', 'NLP_SVD_35', 'NLP_SVD_36', 'NLP_SVD_37', 'NLP_SVD_38', 'NLP_SVD_39', 'NLP_SVD_40', 'NLP_SVD_41', 'NLP_SVD_42', 'NLP_SVD_43', 'NLP_SVD_44', 'NLP_SV

C:\Users\faink\AppData\Local\Temp\ipykernel_13764\981402978.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = features.select_dtypes(include=["object", "category"]).columns.tolist()


In [27]:
# selection features
features_train_baseline = features_train[numeric_cols]
features_test_baseline = features_test[numeric_cols]

In [28]:
# instantiate model
model = LogisticRegression(max_iter=500, class_weight='balanced')

In [29]:
# build pipeline
model_baseline = Pipeline([('model', model)])

In [30]:
# fit pipeline on training set
model_baseline.fit(features_train_baseline, target_train)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# predict and evaluate on test set

target_pred = model_baseline.predict(features_test_baseline)

print("F1-score:", f1_score(target_test, target_pred_grid))
print(classification_report(target_test, target_pred,
                                target_names=["successful tender", "failed tender"]))

In [ ]:
# ---------------------------------------------------------
# Categorical features encoding
# ---------------------------------------------------------

# One-Hot Encoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

# fit on TRAIN
encoder.fit(features_train[categorical_cols])

# final columns
ohe_columns = encoder.get_feature_names(categorical_cols)
final_columns = list(num_cols) + list(categorical_cols)
    
    
def encode(df):
    """
    One-Hot Encoding for train and test.
    
    """
    
    # numeric
    df_num = df[numeric_cols].reset_index(drop=True)

    # transform train/test
    df_ohe = encoder.transform(df[categorical_cols])

    ohe_columns = encoder.get_feature_names(categorical_cols)

    # final dataframe
    df_cat = pd.DataFrame(df_ohe, columns=ohe_columns)
    df_encoded = pd.concat([df_num, df_cat], axis=1)
   
    return df_encoded

In [ ]:
features_train = encode(features_train)
features_test = encode(features_test)

In [14]:
# Preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)


In [15]:
# Model

model = RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)


In [17]:
# Train model

clf.fit(features_train, target_train)


ValueError: Input X contains NaN.
RandomForestClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# Evaluate

target_pred = clf.predict(features_test)
target_proba = clf.predict_proba(features_test)[:, 1]

metrics = {
    "roc_auc": float(roc_auc_score(target_test, target_proba)),
    "f1": float(f1_score(target_test, target_pred)),
    "accuracy": float(accuracy_score(target_test, target_pred)),
}

metrics


In [ ]:
# Save model
joblib.dump(clf, MODEL_PATH)

# Save preprocessor separately (optional)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

# Save feature list
feature_list = {
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "all_feature_cols": features.columns.tolist(),
    "target_col": TARGET_COL,
}

with open(FEATURE_LIST_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_list, f, indent=2)

# Save metrics
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)


In [ ]:
# ROC Curve

plot_roc_curve(
    model=clf,
    features_test=features_test,
    target_test=target_test,
    save=True
)

In [ ]:
# Precision–Recall Curve

plot_pr_curve(
    model=clf,
    features_test=features_test,
    target_test=target_test,
    save=True
)


In [ ]:
# Confusion Matrix

plot_confusion_matrix(
    model=clf,
    features_test=features_test,
    target_test=target_test,
    save=True
)


In [ ]:
# ---------------------------------------------------------
# Feature Importance
# ---------------------------------------------------------

preprocessor = clf.named_steps["preprocessor"]
model = clf.named_steps["model"]

cat_cols = feature_info["categorical_cols"]
num_cols = feature_info["numeric_cols"]

# OneHotEncoder feature names
ohe = preprocessor.named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

# Final list of feature names
feature_names = list(num_cols) + list(cat_feature_names)


In [ ]:
# Plot

plot_feature_importance(
    model=clf,
    feature_names=feature_names,
    save=True
)
